## **A. PREPARATION**

# 1. Install Pyspark

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# 2. SET ENVIRONMENT

In [2]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# 3. Init findspark

In [3]:
import findspark
findspark.init()

# 4. Start SparkSession

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HeartDiseasePipeline") \
    .getOrCreate()

# 5. TEST

In [5]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



# 6. Load dataset

In [6]:
from google.colab import files
files.upload()

Saving heart.csv to heart (2).csv


{'heart (2).csv': b'age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target\r\n52,1,0,125,212,0,1,168,0,1,2,2,3,0\r\n53,1,0,140,203,1,0,155,1,3.1,0,0,3,0\r\n70,1,0,145,174,0,1,125,1,2.6,0,0,3,0\r\n61,1,0,148,203,0,1,161,0,0,2,1,3,0\r\n62,0,0,138,294,1,1,106,0,1.9,1,3,2,0\r\n58,0,0,100,248,0,0,122,0,1,1,0,2,1\r\n58,1,0,114,318,0,2,140,0,4.4,0,3,1,0\r\n55,1,0,160,289,0,0,145,1,0.8,1,1,3,0\r\n46,1,0,120,249,0,0,144,0,0.8,2,0,3,0\r\n54,1,0,122,286,0,0,116,1,3.2,1,2,2,0\r\n71,0,0,112,149,0,1,125,0,1.6,1,0,2,1\r\n43,0,0,132,341,1,0,136,1,3,1,0,3,0\r\n34,0,1,118,210,0,1,192,0,0.7,2,0,2,1\r\n51,1,0,140,298,0,1,122,1,4.2,1,3,3,0\r\n52,1,0,128,204,1,1,156,1,1,1,0,0,0\r\n34,0,1,118,210,0,1,192,0,0.7,2,0,2,1\r\n51,0,2,140,308,0,0,142,0,1.5,2,1,2,1\r\n54,1,0,124,266,0,0,109,1,2.2,1,1,3,0\r\n50,0,1,120,244,0,1,162,0,1.1,2,0,2,1\r\n58,1,2,140,211,1,0,165,0,0,2,0,2,1\r\n60,1,2,140,185,0,0,155,0,3,1,0,2,0\r\n67,0,0,106,223,0,1,142,0,0.3,2,2,2,1\r\n45,1,0,104,208,0,0,148,1,3,1,0,

# 7. Cek struktur data

In [7]:
df = spark.read.csv("heart.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
| 52|  1|  0|     125| 212|  0|      1|    168|    0|    1.0|    2|  2|   3|     0|
| 53|  1|  0|     140| 203|  1|      0|    155|    1|    3.1|    0|  0|   3|     0|
| 70|  1|  0|     145| 174|  0|      1|    125|    1|    2.6|    0|  0|   3|     0|
| 61|  1|  0|     148| 203|  0|      1|    161|    0|    0.0|    2|  1|   3|     0|
| 62|  0|  0|     138| 294|  1|      1|    106|    0|    1.9|    1|  3|   2|     0|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
only showing top 5 rows

root
 |-- age: integer (nullable = true)
 |-- sex: integer (nullable = true)
 |-- cp: integer (nullable = true)
 |-- trestbps: integer (nullable = true)
 |-- chol: integer (nullable = true)
 |-- fbs: integer (nullable =

# 8. Cek missing value

In [8]:
from pyspark.sql.functions import col, isnan, when, count

df.select([
    count(when(isnan(c) | col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|  0|  0|  0|       0|   0|  0|      0|      0|    0|      0|    0|  0|   0|     0|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+



In [ ]:
# df = df.dropna()

## **B. AGGREGATION**

# a. Jumlah pasien sakit vs tidak

In [9]:
df.groupBy("target").count().show()

+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+



# b. Rata-rata umur

In [10]:
df.groupBy("target").avg("age").show()

+------+------------------+
|target|          avg(age)|
+------+------------------+
|     1| 52.40874524714829|
|     0|56.569138276553105|
+------+------------------+



# c. Rata-rata cholesterol

In [11]:
df.groupBy("target").avg("chol").show()

+------+------------------+
|target|         avg(chol)|
+------+------------------+
|     1|240.97908745247148|
|     0| 251.2925851703407|
+------+------------------+



# d. Kombinasi

In [12]:
df.groupBy("sex", "target").count().show()

+---+------+-----+
|sex|target|count|
+---+------+-----+
|  1|     0|  413|
|  1|     1|  300|
|  0|     0|   86|
|  0|     1|  226|
+---+------+-----+



# Execution Time (baseline)

In [13]:
import time

start = time.time()

df.groupBy("target").count().show()

end = time.time()
print("Execution Time (No Optimization):", end - start)

+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+

Execution Time (No Optimization): 0.4428226947784424


# Storage Level (resource insight)

In [14]:
print("Storage Level:", df.storageLevel)

Storage Level: Serialized 1x Replicated


# Jumlah Partisi (parallelism)

In [15]:
print("Number of Partitions:", df.rdd.getNumPartitions())

Number of Partitions: 1


# CACHE

In [16]:
df.cache()
df.count()

1025

# Execution Time setelah Cache

In [17]:
start = time.time()

df.groupBy("target").count().show()

end = time.time()
print("Execution Time (With Cache):", end - start)

+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+

Execution Time (With Cache): 0.3078896999359131


# Repartition

In [18]:
df_re = df.repartition(4)

start = time.time()

df_re.groupBy("target").count().show()

end = time.time()
print("Execution Time (Repartition):", end - start)

+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+

Execution Time (Repartition): 0.5035030841827393


# RDD vs DataFrame

# A. Versi DataFrame

In [19]:
df.groupBy("target").count().show()

+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+



# B. Versi RDD

In [20]:
rdd = df.rdd

rdd_result = rdd.map(lambda x: (x['target'], 1)) \
                .reduceByKey(lambda a, b: a + b)

print(rdd_result.collect())

[(0, 499), (1, 526)]


# C. Bandingkan waktu (WAJIB)

In [21]:
import time

# RDD
start = time.time()

rdd.map(lambda x: (x['target'], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .collect()

print("RDD time:", time.time() - start)


# DataFrame
start = time.time()

df.groupBy("target").count().show()

print("DataFrame time:", time.time() - start)

RDD time: 0.39099788665771484
+------+-----+
|target|count|
+------+-----+
|     1|  526|
|     0|  499|
+------+-----+

DataFrame time: 0.2564702033996582


## **C. MLlib pyspark**

# 1. Feature Vector

In [22]:
features = df.columns
features.remove("target")

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=features, outputCol="features")
df_ml = assembler.transform(df).select("features", "target")

# 2. Split Data

In [23]:
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

# 3. MODEL 1 — Logistic Regression

In [24]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="target", featuresCol="features")
lr_model = lr.fit(train)

lr_pred = lr_model.transform(test)

# 4. MODEL 2 — Random Forest

In [25]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="target", featuresCol="features")
rf_model = rf.fit(train)

rf_pred = rf_model.transform(test)

# 5. MODEL 3 — Decision Tree

In [26]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(labelCol="target", featuresCol="features")
dt_model = dt.fit(train)

dt_pred = dt_model.transform(test)

## **D. EVALUASI**

# 1. Accuracy, Precision, Recall, F1

In [30]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(pred):
    metrics = {}

    metrics["accuracy"] = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="accuracy"
    ).evaluate(pred)

    metrics["precision"] = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="weightedPrecision"
    ).evaluate(pred)

    metrics["recall"] = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="weightedRecall"
    ).evaluate(pred)

    metrics["f1"] = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(pred)

    return metrics

In [32]:
lr_metrics = evaluate_model(lr_pred)
rf_metrics = evaluate_model(rf_pred)
dt_metrics = evaluate_model(dt_pred)

In [33]:
print("=== Logistic Regression ===", lr_metrics)
print("=== Random Forest ===", rf_metrics)
print("=== Decision Tree ===", dt_metrics)

=== Logistic Regression === {'accuracy': 0.8579881656804734, 'precision': 0.8663476991236188, 'recall': 0.8579881656804733, 'f1': 0.8578091090106953}
=== Random Forest === {'accuracy': 0.9408284023668639, 'precision': 0.9432755101871639, 'recall': 0.9408284023668639, 'f1': 0.9408657058397344}
=== Decision Tree === {'accuracy': 0.9053254437869822, 'precision': 0.9056721458869353, 'recall': 0.9053254437869822, 'f1': 0.9053719310129567}


#2. AUC

In [28]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(labelCol="target")

auc_lr = evaluator_auc.evaluate(lr_pred)
auc_rf = evaluator_auc.evaluate(rf_pred)
auc_dt = evaluator_auc.evaluate(dt_pred)

print("AUC LR:", auc_lr)
print("AUC RF:", auc_rf)
print("AUC DT:", auc_dt)

AUC LR: 0.9268258426966294
AUC RF: 0.9837078651685393
AUC DT: 0.9426966292134831


3. waktu training

In [29]:
import time

start = time.time()
lr_model = lr.fit(train)
print("LR time:", time.time() - start)

start = time.time()
rf_model = rf.fit(train)
print("RF time:", time.time() - start)

start = time.time()
dt_model = dt.fit(train)
print("DT time:", time.time() - start)

LR time: 1.9717073440551758
RF time: 1.641817569732666
DT time: 1.8928732872009277
